# Himalayan-GlacierNet: Model Training & Evaluation on Google Colab

This notebook runs the **v1 production pipeline**:
- **Architecture:** `AdapterUNet` (18-to-3 learned adapter stem + pretrained ResNet-34 backbone)
- **Validation:** Leave-One-Region-Out (LORO) zero-leakage cross-validation
- **Optimization:** Automatic Mixed Precision (AMP FP16) on Colab T4 GPU
- **Persistence:** Checkpoints and evaluation reports saved directly to Google Drive

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Check GPU Hardware

In [ ]:
!nvidia-smi

## 3. Clone Repository & Install Dependencies

In [ ]:
import os

# Clone or pull latest main branch
if not os.path.exists('/content/Glacier-Mapping'):
    !git clone https://github.com/MukeshK17/Glacier-Mapping.git /content/Glacier-Mapping
else:
    %cd /content/Glacier-Mapping
    !git pull origin main

%cd /content/Glacier-Mapping
!pip install -q -r requirements.txt
!pip install -q onnxscript

## 4. Verify Google Drive Dataset Location
Locate `datasets128` in your mounted Drive.

In [ ]:
import os

# Search for datasets128 across common Drive locations
candidate_paths = [
    '/content/drive/MyDrive/datasets128',
    '/content/drive/MyDrive/Glacier/datasets128',
    '/content/drive/MyDrive/Glacier-Mapping/datasets128',
]

DATA_DIR = None
for p in candidate_paths:
    if os.path.exists(p):
        DATA_DIR = p
        break

if DATA_DIR is None:
    # Fallback search inside MyDrive
    for root, dirs, _ in os.walk('/content/drive/MyDrive'):
        if 'datasets128' in dirs:
            DATA_DIR = os.path.join(root, 'datasets128')
            break

print(f"Target DATA_DIR found at: {DATA_DIR}")
if DATA_DIR and os.path.exists(DATA_DIR):
    print("Regions detected:", os.listdir(DATA_DIR))
else:
    print("Please check the folder name or update DATA_DIR manually to your Drive path.")

## 5. Launch Training (AdapterUNet with LORO)
Here we train on 4 regions and evaluate zero-shot on **Sikkim** (the hardest debris-covered region).
Checkpoints and metrics are written directly to Google Drive so progress is never lost.

In [ ]:
CHECKPOINT_DIR = '/content/drive/MyDrive/glacier_checkpoints/loro_sikkim'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

!python scripts/train_loro.py \
    --data_dir "$DATA_DIR" \
    --test_region sikkim \
    --model adapter_unet \
    --backbone resnet34 \
    --loss focal_dice_tversky \
    --epochs 35 \
    --batch_size 16 \
    --lr 1e-4 \
    --amp \
    --save_dir "$CHECKPOINT_DIR" \
    --benchmark

### If Colab Disconnects, Resume Seamlessly With:
```bash
!python scripts/train_loro.py \
    --data_dir "$DATA_DIR" \
    --test_region sikkim \
    --resume "$CHECKPOINT_DIR/last_checkpoint.pth"
```

## 6. View Evaluation Results

In [ ]:
import json
results_path = os.path.join(CHECKPOINT_DIR, 'loro_results_sikkim.json')
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print("\n--- Zero-Shot Test Metrics on SIKKIM ---")
    for k, v in results['metrics'].items():
        print(f"{k.upper():12s}: {v:.4f}")
else:
    print("Results file not found yet. Run training cell first.")

## 7. Export Best Model to ONNX and Verify Parity

In [ ]:
from glacier_mapping.models.AdapterUNet import AdapterUNet
from glacier_mapping.utils.benchmark import export_and_verify_onnx, run_benchmark_report
import torch

weights_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
model = AdapterUNet(in_channels=18, backbone='resnet34', pretrained=False)
model.load_state_dict(torch.load(weights_path, map_location='cpu'))

onnx_output = os.path.join(CHECKPOINT_DIR, 'adapter_unet_resnet34.onnx')
export_and_verify_onnx(model, onnx_path=onnx_output)
run_benchmark_report(model, model_name="AdapterUNet (ResNet-34)")